# 1. Extracción, Enriquecimiento e Inteligencia Científica de Aditivos

Este proyecto implementa un flujo de trabajo avanzado de **Ingeniería de Datos y Minería de Textos Científicos** para transformar una lista básica de aditivos alimentarios en un dataset multidimensional. El objetivo es cuantificar la evidencia científica asociada a riesgos y beneficios para la salud mediante el uso de APIs externas.

---

##  Fases del Proceso

### 1. Extracción de Taxonomía (Open Food Facts)
La base del proyecto es la obtención de datos normalizados sobre sustancias alimentarias.
* **Conexión a la API**: Se accede a la taxonomía oficial de *Open Food Facts* para obtener un catálogo global de aditivos.
* **Filtrado mediante Regex**: La función `generar_maestro_aditivos_raw` identifica códigos que cumplen exclusivamente el estándar europeo (ej. "E300" para ácido ascórbico).
* **Gestión de descartes**: Se auditan los elementos que no cumplen el patrón (nombres químicos sin código E o categorías generales) para asegurar la integridad del "maestro" de aditivos.

In [ ]:
import numpy as np
import os
import pandas as pd
import pubchempy as pcp
import re
import requests
import time
import urllib.parse
from tqdm import tqdm

In [3]:
def generar_maestro_aditivos_raw():
    
    """
    Descarga y procesa la taxonomía de aditivos de Open Food Facts para generar un
    catálogo de códigos E (aditivos alimentarios aceptados).

    La función se conecta a la API pública de Open Food Facts, obtiene el diccionario
    completo de aditivos y filtra solo aquellos códigos que cumplen con el patrón
    estándar de los aditivos europeos (por ejemplo, 'e300' para ácido ascórbico).

    """
    
    url = "https://world.openfoodfacts.org/data/taxonomies/additives.json"
    headers = {'User-Agent': 'MyDataScienceProject/1.0 (mklanibarro@gmail.com)'}
    
    print("Conectando con Open Food Facts...")
    try:
        response = requests.get(url, headers=headers, timeout=30)
        if response.status_code == 200:
            data = response.json()
            lista_limpia = []
            diccionario_descartes = {}

            # Regla para Regex: Empieza por 'e', seguido de 1 a 7 caracteres alfanuméricos
            patron = re.compile(r'^e[0-9a-z]{1,7}$', re.IGNORECASE)

            for key, value in data.items():
                # Extraemos el ID corto (ej: de 'en:e300' a 'e300')
                id_corto = key.split(':')[-1] if ':' in key else key
                nombre_en = value.get('name', {}).get('en', 'Desconocido')

                if patron.match(id_corto):
                    lista_limpia.append({
                        'id': key,
                        'codigo_e': id_corto.upper(),
                        'nombre': nombre_en
                    })
                else:
                    # Todo lo que no cumpla la regla va al diccionario de descartes
                    diccionario_descartes[key] = nombre_en
            
            # Crear DataFrame y guardar
            df = pd.DataFrame(lista_limpia)
            df.to_csv('../data/maestro_aditivos_raw.csv', index=False)
            
            print(f"✅ ¡Hecho! Archivo 'maestro_aditivos_raw.csv' guardado.")
            print(f"📊 Aditivos válidos: {len(df)}")
            print(f"📦 Elementos descartados: {len(diccionario_descartes)}")
            
            return df, diccionario_descartes
        else:
            print(f"❌ Error de conexión: Código {response.status_code}")
    except Exception as e:
        print(f"❌ Error inesperado: {e}")
    
    return None, None

# Ejecución
df_final, descartes = generar_maestro_aditivos_raw()

Conectando con Open Food Facts...
✅ ¡Hecho! Archivo 'maestro_aditivos_raw.csv' guardado.
📊 Aditivos válidos: 651
📦 Elementos descartados: 30


### 2. Limpieza y Normalización
Refinamiento de los datos crudos para garantizar búsquedas precisas en bases de datos externas.
* **Estandarización**: Conversión de los códigos `codigo_e` a mayúsculas y limpieza de caracteres especiales.
* **Refinamiento de Nombres**: Separación de códigos y nombres descriptivos para obtener términos químicos limpios (ej. extraer "Dicalcium citrate" de cadenas complejas).

In [ ]:
"""
AUDITORÍA DE DESCARTES:
1. Estructura los datos descartados en un DataFrame para facilitar su análisis.
2. Inspecciona visualmente los primeros registros para detectar patrones de ruido.
3. Filtra nombres científicos específicos que podrían ser aditivos válidos sin código E 
   identificado, asegurando que no se pierda evidencia científica relevante.
"""

# 1. Convertir a DataFrame para poder filtrar
df_descartes = pd.DataFrame(list(descartes.items()), columns=['ID_OFF', 'Nombre_Cientifico'])

# 2. Visualización rápida de los primeros 30
print("--- Primeros elementos descartados ---")
display(df_descartes.head(30))

# 3. Búsqueda de posibles 'Falsos Negativos' 
# (Aditivos reales que no tienen código E pero sí nombre químico)
falsos_negativos = df_descartes[df_descartes['Nombre_Cientifico'].str.contains('acid|carbonate|phosphate', case=False, na=False)]
print("\n--- Posibles aditivos detectados por nombre químico en los descartes ---")
display(falsos_negativos.head(20))

--- Primeros elementos descartados ---


,ID_OFF,Nombre_Cientifico
0,en:methylsulfonylmethane,Methylsulfonylmethane - dimethyl sulfone
1,en:no9,No9 - n9
2,en:no8,No8 - n8
3,en:organic-acids,Organic acids
4,en:fd-c,FD&C - FD and C
5,en:no4,No4 - n4
6,de:natürliche-quellkohlensäure,Desconocido
7,en:no6,No6 - n6
8,en:amino-acids,Amino acids
9,en:n,N° - no



--- Posibles aditivos detectados por nombre químico en los descartes ---


,ID_OFF,Nombre_Cientifico
3,en:organic-acids,Organic acids
8,en:amino-acids,Amino acids
11,en:diphosphates-and-sodium-carbonates,Diphosphates and sodium carbonates
12,en:di-and-triphosphates,Di- and triphosphates
13,en:nucleic-acids,Nucleic acids
22,en:mono-and-diglycerides-of-vegetable-fatty-acids,Mono- and diglycerides of vegetable fatty acids
28,en:ammonium-and-sodium-carbonates,Ammonium and sodium carbonates


El descarte de aditivos ha sido exitoso ya que se han descartado grupos genéricos o confusos

In [ ]:
"""
LIMPIEZA Y ESTANDARIZACIÓN:
1. Formatea la columna 'codigo_e' eliminando prefijos de idioma (en:, fr:) y normalizando a mayúsculas.
2. Refina la nomenclatura eliminando códigos repetidos dentro del nombre para dejar solo el término químico.
3. Exporta un archivo maestro depurado que servirá como base única de verdad para el enriquecimiento.
"""

# Cargar el CSV sucio
df_maestro = pd.read_csv('../data/maestro_aditivos_raw.csv')

# 1. Extraer el código E (E250, E330...)
# Suponiendo que el formato es "en:e250" en la columna 'id'
df_maestro['codigo_e'] = df_maestro['id'].str.split(':').str[-1].str.upper()

# 2. Limpiar la columna nombre
# Si el nombre viene como "E333ii - Dicalcium citrate", cortamos por el guion
df_maestro['nombre'] = df_maestro['nombre'].str.split(' - ').str[-1]

# 3. Guardar el maestro reluciente
df_maestro[['id', 'codigo_e', 'nombre']].to_csv('../data/maestro_aditivos_limpio.csv', index=False)

> **Resultado Final**: Una lista de todos los aditivos alimentarios existentes en la base de datos de Open Food Facts.